In [1]:
import pathlib
import textwrap

import numpy as np
import pickle
from PIL import Image

from IPython.display import display
from IPython.display import Markdown

# from matplotlib.pyplot import imshow

# from sklearn.metrics import roc_auc_score
import re
import PIL.Image
import json
import os
import pandas as pd


def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

from google import genai

In [2]:
all_files = []
img_dir = """/projects/matsci/vlm_microscopy/Microscopy/BBBC005/sampled_images/w2/images"""
for path, subdirs, files in os.walk(img_dir):
    for name in files:
        all_files.append(os.path.join(path, name))

In [ ]:
client = genai.Client(api_key='XYZ')

In [4]:
print("List of models that support generateContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(m.name)

List of models that support generateContent:

models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models

In [5]:
safety_settings = [{"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                   {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"}, 
                   {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                   {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}]

generation_settings = {"top_p": 0.7, "max_output_tokens": 1024}

In [6]:
all_files[-1]

'/projects/matsci/vlm_microscopy/Microscopy/BBBC005/sampled_images/w2/images/SIMCEPImages_A22_C91_F1_s14_w2.png'

In [7]:
last_processed_idx = -1
unprocessed_ids = []
responses = []

In [8]:
import time

In [47]:
for idx, file in enumerate(all_files):
    if idx <= last_processed_idx:
        continue
    img = PIL.Image.open(file)
    print(f"Processing image: {file}")
    prompt = """This is an SEM image of cells. Please count the number of cells in this image in the format 'Count: <count>'. If you are unable to count the cells, please write NaN."""
    response = client.models.generate_content(model = 'gemini-2.5-flash', contents = [img, prompt])
    try:
        answer = response.text
    except Exception as e:
        unprocessed_ids.append(path)
        answer = "MODEL ERROR"
    print(answer)
    print()
    responses.append(answer)
    last_processed_idx = idx
    time.sleep(15)

Processing image: /projects/matsci/vlm_microscopy/Microscopy/BBBC005/sampled_images/w2/images/SIMCEPImages_A08_C31_F1_s10_w2.png


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}

In [11]:
# f8365919f611dcc61903c5ef7093ba6f

In [12]:
len(responses)

35

In [26]:
response.candidates

[Candidate(
   content=Content(
     parts=[
       Part(
         text='Count: 29'
       ),
     ],
     role='model'
   ),
   finish_reason=<FinishReason.STOP: 'STOP'>,
   index=0
 )]

In [27]:
response.prompt_feedback

In [28]:
len(unprocessed_ids)

0

In [29]:
with open('results_BBBC_w2_sampled_counting.pkl', 'wb') as f:
    pickle.dump({"responses": responses, "unprocessed_ids": unprocessed_ids}, f)

In [30]:
with open('results_BBBC_w2_sampled_counting.pkl', 'rb') as f:
    data = pickle.load(f)
    responses = data['responses']

In [31]:
len(unprocessed_ids)

0

In [37]:
unprocessed_ids

[]

In [39]:
actuals = []
predictions = []

for file in all_files:
    actuals.append(int(file.split("_")[-4][1:]))
    
import re

predictions = []
for idx, path in enumerate(all_files):
    if idx < len(responses):
        numbers = re.findall(r'\d+', responses[idx])
    else:
        numbers = ""
    pred = -1
    if len(numbers) > 0:
        pred = int(numbers[0])
    predictions.append(pred)

In [40]:
len(actuals)

50

In [41]:
len(predictions)

50

In [43]:
responses = responses + [""]*15

In [44]:
import pandas as pd
df = pd.DataFrame({'img_path': all_files, 'raw_predictions': responses, 'actuals': actuals, 'predictions': predictions})

In [45]:
df.to_csv('counting_BBBC_gemini_sampled_data_w2.csv')